# Skin Disease CNN — Scratch + EfficientNet Fine-Tuning

Dataset: HAM10000 (Kaggle `kmader/skin-cancer-mnist-ham10000`). Place images at `../data/skin/images/` and the CSV at `../data/skin/HAM10000_metadata.csv`.

In [ ]:
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mn_pre
%matplotlib inline

In [ ]:
DATA = '../data/skin'; IMG = 96
meta_path = os.path.join(DATA, 'HAM10000_metadata.csv')
if os.path.exists(meta_path):
    meta = pd.read_csv(meta_path)
    print('Class distribution:'); print(meta['dx'].value_counts())
    # ... (load images by image_id, see notebook expansion in repo wiki)
    raise NotImplementedError('Plug in your HAM10000 image loader here.')
else:
    print('HAM10000 not found — falling back to synthetic patches.')
    import sys; sys.path.insert(0, '../scripts')
    from train_skin import synth_images, CLASSES
    X, y = synth_images(n_per_class=400)
    X = tf.image.resize(X, (IMG, IMG)).numpy()
    split = int(0.8*len(X))
    train_ds = tf.data.Dataset.from_tensor_slices((X[:split]*255, y[:split])).shuffle(1024).batch(32)
    val_ds = tf.data.Dataset.from_tensor_slices((X[split:]*255, y[split:])).batch(32)
    class_names = CLASSES
print(class_names)

## Look at samples

In [ ]:
for imgs, lbls in train_ds.take(1):
    fig, axes = plt.subplots(2, 5, figsize=(12,5))
    for ax, im, l in zip(axes.flat, imgs, lbls):
        ax.imshow(np.array(im).astype('uint8')); ax.set_title(class_names[int(l)], fontsize=8); ax.axis('off')
    plt.show()

## Model A: small CNN from scratch

In [ ]:
def make_scratch():
    m = models.Sequential([
        layers.Input((IMG, IMG, 3)),
        layers.Rescaling(1./255),
        layers.RandomFlip('horizontal'), layers.RandomRotation(0.1), layers.RandomZoom(0.1),
        layers.Conv2D(32, 3, padding='same', activation='relu'), layers.MaxPool2D(),
        layers.Conv2D(64, 3, padding='same', activation='relu'), layers.MaxPool2D(),
        layers.Conv2D(128, 3, padding='same', activation='relu'), layers.MaxPool2D(),
        layers.GlobalAveragePooling2D(), layers.Dropout(0.4),
        layers.Dense(64, activation='relu'),
        layers.Dense(len(class_names), activation='softmax'),
    ])
    m.compile(optimizer=optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m
scratch = make_scratch()
es = callbacks.EarlyStopping(patience=3, restore_best_weights=True)
hist = scratch.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[es])

In [ ]:
plt.plot(hist.history['accuracy'], label='train'); plt.plot(hist.history['val_accuracy'], label='val')
plt.legend(); plt.title('Scratch CNN'); plt.show()

## Model B: MobileNetV2 transfer learning + fine-tune

In [ ]:
base = MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False
tl = models.Sequential([
    layers.Input((IMG, IMG, 3)), layers.Lambda(mn_pre), base,
    layers.GlobalAveragePooling2D(), layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax'),
])
tl.compile(optimizer=optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
tl.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[es])
# fine-tune
base.trainable = True
for l in base.layers[:-40]: l.trainable = False
tl.compile(optimizer=optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
tl.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[es])

In [ ]:
os.makedirs('../models', exist_ok=True)
scratch.save('../models/skin_cnn.h5')
with open('../models/skin_labels.json', 'w') as f: json.dump(list(class_names), f)
print('saved.')